# 08 · Summary statistics

The summary statistics of the flux, built bottom-up on toy boxes whose statistics are known.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag)
from skewerjax.core import LineBundle

In [ ]:
plot_dag(('lines', 'cosmology', 'skewers', 'samplers', 'fields', 'profiles', 'deposition', 'tau', 'overflux', 'stats',)); plt.show()

## Concepts

**3D two-point statistics on a grid**. A field $\delta(\mathbf x)$ on
$N_x \times N_y \times N_z$ points with spacings $\Delta_i$ (the units may differ per axis). We :

$$
L_i = N_i\Delta_i,\qquad V = \prod_i L_i,\qquad \Delta V = \prod_i \Delta_i,\qquad N = \prod_i N_i .
$$

The DFT, on the modes $k_i = 2\pi m_i/L_i$ with $m_i \in (-N_i/2,\, N_i/2]$:

$$
\delta_{\mathbf k} = \Delta V\sum_{\mathbf x}\delta(\mathbf x)\,e^{-i\mathbf k\cdot\mathbf x}.
$$

The estimators, both without the $\mathbf k = 0$ mode:

$$
\hat P(\mathbf k) = \frac{\lvert\delta_{\mathbf k}\rvert^2}{V}\quad(\mathbf k \neq 0),\qquad
\hat\xi(\mathbf r) = \frac{1}{V}\sum_{\mathbf k \neq 0}\hat P(\mathbf k)\,e^{i\mathbf k\cdot\mathbf r}
= \frac{1}{N}\sum_{\mathbf x} d(\mathbf x)\,d(\mathbf x + \mathbf r),\qquad d = \delta - \frac{1}{N}\sum_{\mathbf x}\delta(\mathbf x),
$$

with $\mathbf r$ every minimum-image separation of the periodic grid ($\lvert r_i\rvert \le L_i/2$).

The binning:

$$
k_\parallel = \lvert k_y\rvert,\qquad k_\perp = \sqrt{k_x^2 + k_z^2},\qquad
\bar P\big(k_\perp^{(a)}, k_\parallel\big) = \frac{1}{N_a}\sum_{\mathbf k \in a}\hat P(\mathbf k),
$$

- $k_\parallel$ keeps the native modes $2\pi m/L_y$ (one row each, no binning);
- $k_\perp$ is averaged within given edges, as above, over the $N_a$ modes of bin $a$;
- $\lvert\mathbf k\rvert$ bins, and wedges of $\mu = k_\parallel/\lvert\mathbf k\rvert$ in $\lvert\mathbf k\rvert$ bins,
  in the same way;
- identically in real space, with $r_\parallel = \lvert r_y\rvert$ (native separations $j\Delta_y$),
  $r_\perp = \sqrt{r_x^2 + r_z^2}$, $\lvert\mathbf r\rvert$ and $\mu = r_\parallel/\lvert\mathbf r\rvert$.

**Errors:** the error of the mean over the $n_{\rm real}$ realisations, $\sigma_a/\sqrt{n_{\rm real}}$ per bin.

**Overview.**

- **Part A — toy fields:** `n_real` periodic boxes with the spectrum and gas of the toy model below; $P$ and $\xi$ of the
  realisations against their expectation and against the continuous $P(k)$ and $\xi(r)$.
- **Part B — skewers:** a regular lattice of skewers along $y$ through every box (`SkewerRun`); $\tau$ of
  $\mathrm{Ly}\alpha$ and Si III, periodic.
- **Part C — 3D statistics of the overfluxes** on the lattice of skewers: the $(r_\perp, r_\parallel)$ plane, $P$ and
  $r^2\xi$, the wedges of $\mu$.
- **Part D — 1D statistics of the overfluxes:** P1D next to a measurement, order by order and term by term.
- **Parts E and F — the same statistics of the density** on the same skewers, figure by figure, against their exact
  expectation.

## The toy model: what `ToyBoxes` generates

**Grid and realisations.** $n_{\rm real}$ periodic cubes of side $L$ and volume $V = L^3$ on $N^3$ cells of side
$dx = L/N$; the modes are $\mathbf k = (2\pi/L)\,\mathbf n$ with integers $-N/2 < n_i \le N/2$. Realisation $i$ is drawn
from the seed `base_seed` $+\,i$ and regenerated whenever it is needed: white noise $w$ on the
grid, $\delta_{\mathbf k} = dV\,\mathrm{FFT}[w]_{\mathbf k}\sqrt{P(k)/dV}$, so that
$\langle\lvert\delta_{\mathbf k}\rvert^2\rangle = V P(k)$, with $\delta_{\mathbf k = 0} = 0$.

**Correlation function and power spectrum** (isotropic: no Kaiser or Fingers-of-God term). The simplest model
with a BAO feature: a power law plus a Gaussian bump at $r_{\rm BAO}$,

$$
\xi(r) = \underbrace{\frac{A_s\,k_0^{1 - n_s}}{2\pi^2}\,
\frac{\Gamma(n_s + 1)\,\sin\big(\pi(n_s + 1)/2\big)}{r^{\,n_s + 2}}}_{\xi_{\rm pl}(r) \equiv \text{power law}}
+ \underbrace{A_{\rm BAO}\big[g(r - r_{\rm BAO}) + g(r + r_{\rm BAO})\big]}_{\text{bump}},\qquad
g(u) = e^{-u^2/2\sigma_{\rm BAO}^2},\qquad A_{\rm BAO} = R_{\rm BAO}A_s,
$$

$$
P(k) = \underbrace{A_s\left(\frac{k}{k_0}\right)^{n_s - 1}}_{\text{power law}}
+ \underbrace{A_{\rm BAO}\,W(k)}_{\text{bump}},\qquad
W^{\rm 3D}(k) = 4\pi\sqrt{2\pi}\,\sigma_{\rm BAO}\,e^{-k^2\sigma_{\rm BAO}^2/2}
\left[\sigma_{\rm BAO}^2\cos(kr_{\rm BAO}) + \frac{r_{\rm BAO}}{k}\sin(kr_{\rm BAO})\right],
$$

$W$ being the exact Fourier transform of the bump (the mirror term $g(r + r_{\rm BAO})$ makes it even). In 1D,
$W^{\rm 1D} = 2\sqrt{2\pi}\,\sigma_{\rm BAO}\,e^{-k^2\sigma_{\rm BAO}^2/2}\cos(kr_{\rm BAO})$; in 2D, the Hankel
transform with $J_0$, by quadrature.

**Validity.** A valid correlation function needs $P(k) \ge 0$ at every mode (Bochner's theorem; equivalently
$\lvert\xi(r)\rvert \le \xi(0)$). $W$ changes sign (a bump away from $r = 0$ alone is not a valid correlation function),
so the power law must exceed the troughs of $A_{\rm BAO}W$:

$$
R_{\rm BAO} \le R_{\max} = \min_{\mathbf k \in \text{grid},\ W(k) < 0}\frac{(k/k_0)^{n_s - 1}}{\lvert W(k)\rvert},
$$

evaluated on the modes of the box and independent of $\sigma_\delta$ (`ToyBoxes` raises an error quoting $R_{\max}$ for a larger `bao_ratio`).

**The amplitude.** $A_s$ is set by $\sigma_\delta$, the standard deviation of $\delta$ over the grid. At fixed $R_{\rm BAO}$ both terms are proportional to $A_s$, $P(k) = A_s\,p(k)$ with the shape
$p(k) = (k/k_0)^{n_s - 1} + R_{\rm BAO}W(k)$, and the variance of the grid is the sum of $P$ over its modes:

$$
\sigma_\delta^2 = \frac{1}{V}\sum_{\mathbf k \neq 0} P(k) = A_s\,S,\qquad
S = \frac{1}{L^3}\sum_{\substack{\mathbf n \neq 0\\ -N/2 < n_i \le N/2}} p\!\left(\frac{2\pi\lvert\mathbf n\rvert}{L}\right),
\qquad A_s = \frac{\sigma_\delta^2}{S}.
$$

$S$ is computed from the shape alone (`PowerLawGRF.variance` at $A_s = 1$), and $A_s$ is the ratio.

**The continuous correlation function** $\xi(r) = (2\pi^2 r)^{-1}\int_0^\infty k\,P(k)\sin(kr)\,dk$ of this spectrum, in
closed form term by term:

$$
\xi(r) = \underbrace{\frac{A_s\,k_0^{1 - n_s}}{2\pi^2}\,
\frac{\Gamma(n_s + 1)\,\sin\big(\pi(n_s + 1)/2\big)}{r^{\,n_s + 2}}}_{\xi_{\rm pl}:\ \text{power law}}
+ \underbrace{A_{\rm BAO}\big[g(r - r_{\rm BAO}) + g(r + r_{\rm BAO})\big]}_{\text{bump}} .
$$

Regarding the term $\int_0^\infty x^{\mu - 1}\sin x\,dx = \Gamma(\mu)\sin(\pi\mu/2)$ with $\mu = n_s + 1$: The integral
converges for $-1 < \mu < 1$, so **the closed form holds for $-2 < n_s < 0$**.
Outside that range the continuous $\xi$ of the power law does not exist: the
integral diverges at $k \to 0$ for $n_s \le -2$ and at $k \to \infty$ for $n_s \ge 0$. `PowerLawGRF.correlation` then
says so in one line and returns the numerical Hankel transform of $P$ over the $k$ range of the grid instead.

**What sets $P$ and $\xi$.** Eight inputs:

| Symbol | `ToyBoxes` argument | Meaning | Enters |
|---|---|---|---|
| $\sigma_\delta$ | `sigma_delta` | standard deviation of $\delta$ on the grid | $A_s = \sigma_\delta^2/S$ [$\mathrm{cMpc}^3$] |
| $k_0$ | `k_0` | pivot wavenumber [cMpc$^{-1}$] | the shape $p$ |
| $n_s$ | `n_s` | spectral index: $P \propto k^{n_s - 1}$ | the shape $p$ |
| $R_{\rm BAO}$ | `bao_ratio` | $A_{\rm BAO}/A_s$, the height of the bump in $\xi$ per unit $A_s$, at most $R_{\max}$ [cMpc$^{-3}$] | the shape $p$ |
| $\sigma_{\rm BAO}$ | `sigma_bao` | width of the bump [cMpc] | the shape $p$ |
| $r_{\rm BAO}$ | `r_bao` | position of the bump [cMpc] | the shape $p$ |
| $L$ | `box_cMpc` | box side [cMpc] | the modes: $S$ (so $A_s$) and $\xi_{\rm grid}$ |
| $N$ | `n_cells` | cells per side | the modes: $S$ (so $A_s$) and $\xi_{\rm grid}$ |

**Gas.** H I and Si III are one gas, cell by cell: $n_{\rm HI} = \bar n_{\rm HI}(1 + \delta)$,
$n_{\rm SiIII} = r\,n_{\rm HI}$, $T_{\rm gas} = T_0(1 + \delta)^{\gamma - 1}$ (written $T_{\rm gas}$ here, as $T$ is
the transmission), and peculiar velocities $\mathbf v = 0$ or linear theory, $\mathbf v_{\mathbf k} = i\,aHf\,
(\mathbf k/k^2)\,\delta_{\mathbf k}$. The remaining `ToyBoxes` inputs set this gas and its skewers:
`n_bar_HI` ($\bar n_{\rm HI}$, which sets $\langle T_{{\rm Ly}\alpha}\rangle$), `siiii_to_hi` ($r$), `T0`,
`gamma_minus_1`, `velocity` (`"zero"`, the default, or `"linear"`), `f_growth` ($f$, for `"linear"` only), and `z`,
`Omega_m` ($a$ and $H$ of a flat cosmology: the Hubble flow, the velocity axis of the skewers); `n_real` and `base_seed`
choose the realisations.

**Positivity, $1 + \delta > 0$.** A Gaussian $\delta$ is unbounded, but a density $n = \bar n(1 + \delta)$ must be
positive (and $T_{\rm gas}$ needs $1 + \delta > 0$). The densities are not clipped; `ToyBoxes.fields` raises if any cell
has $\delta \le -1$. One cell falls below $-1$ with probability $\Phi(-1/\sigma_\delta)$ ($\Phi$ the standard normal
distribution function), and a box has $N^3$ cells, so about $N^3\,\Phi(-1/\sigma_\delta)$ of them do (printed below): a
finer grid needs a smaller $\sigma_\delta$. This is separate from the validity of the spectrum: $P(k) \ge 0$ at every
mode is the condition $R_{\rm BAO} \le R_{\max}$, independent of $\sigma_\delta$; only the mapping from $\delta$ to a
density needs $1 + \delta > 0$.

In [ ]:
import time
from fractions import Fraction
from skewerjax.core import phys
from skewerjax.simulations import PowerLawGRF, ToyBoxes, field_statistics, xi_chi2, xi_subset_offsets
from skewerjax.simulations.toy_boxes import LINE_IDS
REALISATIONS = None               # None: all of them; e.g. [0, 1] for a quick check
L_BOX = 150
SIIII_TO_HI = 1 / 10 # n_SiIII = r n_HI
boxes = ToyBoxes(box_cMpc=L_BOX, n_s = -0.4, n_cells=128, n_real=6, base_seed=42, bao_ratio=0.003, sigma_bao=1.0, r_bao=40, siiii_to_hi=SIIII_TO_HI, velocity="zero", z = 3.0)

In [ ]:
# The values of the configuration (the text refers to them by their symbols)
from scipy.stats import norm
g = boxes.grf()
idx = np.arange(boxes.n_real) if REALISATIONS is None else np.asarray(REALISATIONS)
var_bao = 1.0 - float(PowerLawGRF(g.A_s, g.k_0, g.n_s, L=g.L, N=g.shape).variance() / g.variance())
S_shape = boxes.sigma_delta**2 / boxes.A_s                                       # the mode sum of the shape p(k), A_s = 1
print(f"dx = {boxes.dx:.4g} cMpc ({boxes.dx * boxes.aH:.1f} km/s); R_BAO = {boxes.bao_ratio:g} <= R_max = {boxes.bao_ratio_max:.4g} cMpc^-3 (P >= 0 at every mode of the grid): A_BAO = {g.A_bao:.4g}, A_s = {g.A_s:.4g} cMpc^3")

## Part A — Toy fields and their statistics

### A1 · The fields and their bins

**Where the statistics are computed.** The $P$ and $\xi$ of every realisation are computed in the `field_statistics(...)`
cell below: each realisation is generated, measured and discarded in one pass.

**Coverage** (printed after `field_statistics`). The grid fixes the ranges where a bin holds every direction:

- $\lvert\mathbf k\rvert$ from $k_{\min} = 2\pi/L$ to $k_{\max} = \pi/dx$, the Nyquist frequency of each axis. Inside
  the sphere of radius $\pi/dx$ a shell of $\lvert\mathbf k\rvert$ has modes in every direction; beyond it only the
  corners of the cube of modes remain.
- $\lvert\mathbf r\rvert$ from $r_{\min} = dx$ to $r_{\max} = L/2$. Beyond half the box only the separations towards the
  corners of the cube remain (minimum image).

If the BAO peak, $r_{\rm BAO} \pm 4\sigma_{\rm BAO}$,
reaches beyond $L/2$, a one-line warning says that part of it lies outside the full-coverage range.

**The bins** are explicit arrays, defined in the next cell and passed to `field_statistics`:

- `K_EDGES`, the bins of $P(\lvert\mathbf k\rvert)$. One bin per mode up to $k_1$; then bins of a
  quarter of the fundamental mode $k_f = 2\pi/L$, up to $k_2 \simeq 3/\sigma_{\rm BAO}$; then logarithmic bins from $k_2$ up to $k_{\max}$.
  `K_EDGES` are also the $k_\perp$ columns of the $(k_\perp, k_\parallel)$ map.
- `R_EDGES`, the bins of $\xi(\lvert\mathbf r\rvert)$: width $dx$, from 0 to $L/2$. The grid holds no mode beyond
  $\pi/dx$. The first bin holds $r = 0$ only, the variance, which the figures leave out. `R_EDGES`
  are also the $\lvert\mathbf r\rvert$ bins of the wedges of $\mu$ and the $r_\perp$ columns of the
  $(r_\perp, r_\parallel)$ map.

In [ ]:
# The bins of A3 [cMpc^-1], [cMpc], passed to field_statistics and to the figures (text above)
L, dx = boxes.box_cMpc, boxes.dx
K_EDGES, K_INFO = boxes.k_edges(k_1=None, k_2_factor=3.0, n_log=12, info=True)  # |k|: per mode to k_1, k_f / 4 to k_2, log to pi / dx
R_EDGES = np.arange(0, L / 2 + dx, dx)                                          # |r| (and the mu wedges, the r_perp columns): width dx to L / 2

In [ ]:
res = field_statistics(boxes, k_edges=K_EDGES, r_edges=R_EDGES, realisations=REALISATIONS)  # P and xi of every realisation (A3), computed now
m1 = res["min_one_plus_delta"]
print(f"{res['n_real']} realisations measured in {res['time_s']:.0f} s, and the covariance of xi in {res['time_cov_s']:.0f} s: "
      f"min(1 + delta) = {m1.min():.3f} (realisation {res['realisations'][m1.argmin()]}); 1 + delta > 0 in {np.sum(m1 > 0)} of {m1.size}")
full, ax_A = res["limits"], res["axes"]
print(f"full coverage of the grid: |k| from k_min = 2 pi / L = {full['k_min']:.4f} to k_max = pi / dx = {full['k_max']:.3f} cMpc^-1 "
      f"; |r| from r_min = dx = {full['r_min']:.3f} to r_max = L / 2 = {full['r_max']:.1f} cMpc")
cap = " (capped at k_max)" if K_INFO["k_2"] < K_INFO["k_w"] else ""
print(f"|k| bins: {K_INFO['n_mode']} of one mode up to k_1 = {K_INFO['k_1']:.4f}, {K_INFO['n_quarter']} of k_f / 4 = "
      f"{full['k_min'] / 4:.4f} up to k_2 = {K_INFO['k_2']:.3f}{cap} (factor / sigma_BAO = {K_INFO['k_w']:.3f}), "
      f"{K_INFO['n_log']} logarithmic from k_2 up to {ax_A['k_edges'][-1]:.3f} cMpc^-1")
print(f"|r| bins: {ax_A['r_edges'].size - 1} of width {np.diff(ax_A['r_edges']).max():.3f} cMpc from {ax_A['r_edges'][0]:g} to "
      f"{ax_A['r_edges'][-1]:.1f} cMpc; every bin inside the full coverage: |k| {ax_A['k_edges'][-1] <= full['k_max'] * (1 + 1e-9)}, "
      f"|r| {ax_A['r_edges'][-1] <= full['r_max'] * (1 + 1e-9)}")
if boxes.r_bao + 4 * boxes.sigma_bao > full["r_max"]:
    print(f"WARNING: the BAO peak reaches r_BAO + 4 sigma_BAO = {boxes.r_bao + 4 * boxes.sigma_bao:.1f} cMpc, beyond r_max = L / 2 = "
          f"{full['r_max']:.1f} cMpc: part of the peak lies outside the full-coverage range")

### A2 · The fields of one realisation

Realisation 0 in the plane of the first cell layer along the third axis (the cells with index $i_z = 0$), which holds
the first row of skewers of Part B: $x$ across and $y$, the line of sight of those skewers, upwards. The two columns
differ only by the factor $r$ of the densities; the temperature (not drawn) is a function of the density of the top row.
The bottom row shows the velocity along the skewers, $v_y$ (colour), and $(v_x, v_y)$ as arrows every `ARROW_EVERY`
cells; with zero velocities (`velocity="zero"`) it is empty and says so.

In [ ]:
from skewerjax.utils.plotting import plot_toy_fields
ARROW_EVERY = max(1, boxes.n_cells // 32)                                    # about 32 arrows across the slice
fields0 = boxes.fields(0)
fig, axs, info = plot_toy_fields(fields0, L_BOX, z_index=0, arrow_every=ARROW_EVERY)
add_panel_letters(axs.T.ravel(), labels=["a1", "a2", "b1", "b2"])
plt.show()

### A3 · The statistics of the realisations

Each realisation is measured on the bins of A1 (`K_EDGES`, `R_EDGES`). Panel (a) shows $P(k)$ , with
dotted lines at $k_1$ and $k_2$, where the bins of `K_EDGES` change; panel (b) shows $\xi(r)$.
The grey bands lie outside the full-coverage range of the grid, with its limits written inside them: $k < k_{\min}$
and $k > k_{\max}$ in (a), $r < r_{\min}$ and $r > r_{\max}$ in (b).

**The expectation (black line).**

1. The generator draws every mode $\mathbf k$ of the box with $\langle\hat P(\mathbf k)\rangle = P(k)$, and the
   correlation at every separation $\mathbf r$ of the grid is, on average, $\xi_{\rm grid}(\mathbf r)$.
2. A bin $a$ of $\lvert\mathbf k\rvert$ holds the $N_a$ modes of the box that fall in it, so the expectation of the bin
   average is the input averaged over exactly those modes,
   $\bar P_a = N_a^{-1}\sum_{\mathbf k \in a} P(\lvert\mathbf k\rvert)$.
3. Likewise, a bin $a$ of $\lvert\mathbf r\rvert$ holds the $N_a$ separations of the grid that fall in it, and
   $\bar\xi_a = N_a^{-1}\sum_{\mathbf r \in a}\xi_{\rm grid}(\mathbf r)$, with
   $\xi_{\rm grid}(\mathbf r) = V^{-1}\sum_{\mathbf k \neq 0} P(k)\cos(\mathbf k\cdot\mathbf r)$.

**Continuous $P(k)$ and $\xi(r)$ (purple line).** They are the infinite-volume curves of the same
spectrum, over the full-coverage range, and a finite periodic grid does not measure them:

- **No $k = 0$ mode.** $\xi_{\rm grid}$ integrates to zero over the box, so it lies below $\xi(r)$ by about the mean of
  $\xi$ over the box, a nearly constant offset.
- **Periodic images.** $\xi_{\rm grid}(\mathbf r)$ adds the correlation of the images $\mathbf r + L\mathbf n$.
- **Averaging within bins.** $P$ and $\xi$ vary inside a bin, and the few modes of a low-$k$
  bin sample it unevenly.

The binned expectation includes all of this, so it is exact bin by bin; the continuous curves are for reference only.

**The error bars** are the error of the mean over the $n_{\rm real}$ realisations, bin by bin:
$\sigma_a/\sqrt{n_{\rm real}}$, with $\sigma_a$ the standard deviation (with $n_{\rm real} - 1$) of the $n_{\rm real}$
values of bin $a$. The residual panels divide by it.

In [ ]:
from skewerjax.utils.plotting import plot_isotropic_recovery, plot_kpar_kperp_maps, plot_rpar_rperp_maps
K_MARKS = {"$k_1$": K_INFO["k_1"], "$k_2$": K_INFO["k_2"]}                # where the bins of K_EDGES change (A1)
fig, axs, rec = plot_isotropic_recovery(res, quantity="r2xi", r_bao=boxes.r_bao, grf=g, k_edges=K_EDGES, r_edges=R_EDGES, k_marks=K_MARKS, y_scale="symlog", k_scale="linear")
add_panel_letters(axs[0])                                              # (a) P, (b) r^2 xi; the residual panels below them
plt.show()
SUBSETS = (4, 8, 16)                                                   # disjoint subsets of realisations (reading below)
XI_WINDOW = (2 * boxes.r_bao / 3, boxes.r_bao + 4 * boxes.sigma_bao)   # the r range of the offset [cMpc]
c_all, c_win = xi_chi2(res), xi_chi2(res, r_range=XI_WINDOW)
r_c = np.linspace(*rec["xi_curve_r"], 400)                             # the r range of the continuous xi(r) in (b)
if -2 < g.n_s < 0:
    d_cf = np.max(np.abs(g.correlation(r_c) / g.correlation_hankel(r_c, k_tail=400) - 1))
    print(f"continuous xi(r): the closed form against a numerical Hankel transform of the full P(k), max relative difference "
          f"{d_cf:.1e} for {r_c[0]:.3g} <= r <= {r_c[-1]:.3g} cMpc")
r_i, th_i = ax_A["r_iso"], res["theory"]["xi_iso"]
for r0 in (boxes.r_bao, full["r_max"]):
    j = int(np.argmin(np.abs(r_i - r0)))
    print(f"continuous xi minus the binned expectation at r = {r_i[j]:.1f} cMpc: {float(g.correlation(r_i[j:j + 1])[0]) - th_i[j]:.2e}"
          f" (the missing k = 0 mode)")
print(f"P(|k|): chi2/dof = {rec['P_chi2_dof']:.2f}, max |P / P_th - 1| = {rec['P_max_rel']:.3f}")
w_C = np.linalg.eigvalsh(res["cov_xi_iso"][1:, 1:])
print(f"covariance of the xi bins (r > 0): condition number {w_C.max() / w_C.min():.1e} (float64)")
for name, c in (("all bins", c_all), (f"{XI_WINDOW[0]:.0f} < r < {XI_WINDOW[1]:.0f} cMpc", c_win)):
    print(f"xi(|r|), {name} ({c['dof']} bins): chi2 diagonal = {c['chi2_diag']:.1f}, full covariance = {c['chi2_full']:.1f} "
          f"(p = {c['p_full']:.2f}); first eigenvector {100 * c['mode1_share']:.0f} % of the variance; the offset lies "
          f"{100 * c['offset_top3_share']:.0f} % along the 3 largest")
print(f"chi2 of each realisation (all bins, covariance of one realisation): mean {c_all['chi2_each'].mean():.1f} for {c_all['dof']} bins")
for o in xi_subset_offsets(res, SUBSETS, r_range=XI_WINDOW):
    print(f"offset of r^2 xi in XI_WINDOW, subsets of {o['size']}: " + ", ".join(f"{x:+.3f}" for x in o["offsets"])
          + f" cMpc^2 (standard deviation {o['sigma']:.3f})")

*Reading the figure.* In (b) the points can sit on one side of the expectation over a whole range of $r$, by more than their
error bars. This is one ordinary fluctuation, not a bias:

- Every bin of $\xi$ is a sum over the same modes,
  $\hat\xi(\mathbf r) = V^{-1}\sum_{\mathbf k}\hat P(\mathbf k)\cos(\mathbf k\cdot\mathbf r)$, and with a red spectrum
  ($P(k)$ decreasing with $k$, here $n_s - 1 < 0$, so most of the power is on large scales) the few largest-scale modes
  carry most of the signal. When their random amplitudes come out a high (or low),
  every bin moves up (or down) together. The error bars show the scatter of each bin alone, so they cannot show this:
  several points off by about $1\sigma$ in the same direction are one fluctuation, not several.
- The exact covariance $C$ of the bins is dominated by its first eigenvector. The $\chi^2$ that accounts for it,
  $\chi^2 = \mathbf d^{\rm T}(C/n_{\rm real})^{-1}\mathbf d$ with $\mathbf d = \bar\xi - \xi_{\rm grid}$, is close to
  the number of bins, over all the bins and over the window of the BAO peak, `XI_WINDOW`.
- **More realisations shrink it.** For disjoint subsets of realisations, the offset of $\xi$ in `XI_WINDOW` changes sign from subset to subset and stays within its standard
  deviation, which falls as $1/\sqrt n$.
- **$P$ shows no such effect**, because its bins hold different modes, which are independent.

The purple continuous $\xi(r)$ lies above both: the missing $k = 0$ mode lowers $\xi_{\rm grid}$ by a nearly constant
amount; the purple $P(k)$ runs through the binned expectation, except in the first bins, which hold a few modes each.

**The $(k_\perp, k_\parallel)$ and $(r_\perp, r_\parallel)$ planes.** The same statistics resolved across ($x$ axis) and
along ($y$ axis) the line of sight, on the grid's own modes and separations: in each figure, (a) the expectation and
(b) the mean measured. $P$ on a logarithmic colour scale with both axes logarithmic; $r^2\xi$ on a linear colour scale,
linear axes up to half the box. Three iso-contour levels, the same for the expectation (dashed, in both panels) and the
measurement (green, with its band of $\pm$ `n_sigma` errors). The white rays are the boundaries of the wedges of $\mu$
of the wedge figure (`MU_WEDGES`), and the coloured sectors at the origin their colours there; the dotted circle,
labelled on the panels, is $r = r_{\rm BAO}$.

In [ ]:
fig, axs, chi2_P = plot_kpar_kperp_maps(res, kperp_edges=K_EDGES, n_sigma=1, k_marks=K_MARKS, kpar_scale="linear", kperp_scale="linear")
add_panel_letters(axs)
plt.show()
print(f"chi2/dof per bin: P map {chi2_P:.2f}; {ax_A['kpar_edges'].size - 1} rows (k_par = 2 pi m / L, m = 1 .. N / 2), "
      f"{ax_A['kperp_edges'].size - 1} columns of k_perp (K_EDGES), of which {int(np.sum(ax_A['n_modes_P_map'].sum(0) == 0))} hold no mode")

*The bins.* One row per mode of the grid along the line of sight, $k_\parallel = 2\pi m/L$ from $2\pi/L$ to $\pi/dx$,
each centred on its $k_\parallel$. Across, every mode has $k_\perp = \sqrt{k_x^2 + k_z^2}$ and is averaged into the
column of `K_EDGES` that contains it; the columns span their bin edges, and a column between two allowed values of
$k_\perp$ holds no mode and is left empty, in light grey (printed).

In [ ]:
fig, axs, chi2_xi = plot_rpar_rperp_maps(res, rperp_edges=R_EDGES, r_bao=boxes.r_bao, n_sigma=0.2, quantity="r2xi", color_scale="symlog")
add_panel_letters(axs)
plt.show()
print(f"chi2/dof per bin: xi map {chi2_xi:.2f}; {ax_A['rpar_edges'].size - 1} rows (r_par = j dx, j = 0 .. N / 2), "
      f"{ax_A['rperp_edges'].size - 1} columns of r_perp (R_EDGES)")
# The ring and what lies beneath it, wedge by wedge at r_BAO: the bump = xi - xi_nw (xi_nw: the expectation of the
# power law alone) and the background xi_nw
a = res["axes"]
j = np.nanargmin(np.abs(a["r_mu"] - boxes.r_bao), axis=1)   # empty bins of a narrow wedge are NaN
w = np.arange(len(j))
th, nw = res["theory"]["xi_mu"][w, j], res["theory_nw"]["xi_mu"][w, j]
meas, err = res["xi_mu"][w, j] - nw, res["xi_mu_err"][w, j]
print(f"at r = r_BAO, in units of A_BAO = {g.A_bao:.3g}:")
print("  wedge of mu      bump expected   bump measured     background xi_nw")
for k in w:
    print(f"  {a['mu_edges'][k]:.2f} - {a['mu_edges'][k + 1]:.2f}       {(th[k] - nw[k]) / g.A_bao:6.3f}      "
          f"{meas[k] / g.A_bao:6.2f} +- {err[k] / g.A_bao:4.2f}      {nw[k] / g.A_bao:+6.2f}")
s_diag = np.ones(3) / np.sqrt(3)                                          # a body diagonal (text below)
n_img = np.array(np.meshgrid(*[np.arange(-1, 2)] * 3, indexing="ij")).reshape(3, -1).T
n_img = n_img[np.any(n_img != 0, axis=1)]                                  # the nearest periodic images, n != 0
d_1 = lambda s: np.min(np.linalg.norm(0.5 * s + n_img, axis=1))            # [L], for r = L / 2 along s
print(f"nearest periodic image of r = L / 2: along an axis at {d_1(np.array([0.0, 1.0, 0.0])):.3f} L, along a body diagonal "
      f"at {d_1(s_diag):.3f} L")
k_nyq = np.pi / boxes.dx
print(f"|P_BAO| at the Nyquist frequency pi/dx = {k_nyq:.2f} cMpc^-1, relative to its value at the fundamental mode: "
      f"{abs(float(g.bao_power(k_nyq) / g.bao_power(2 * np.pi / L_BOX))):.1e}")

*The bins.* One row per separation of the grid along the line of sight, $r_\parallel = j\,dx$ from 0 to $L/2$, each
centred on its separation. Across, every pair of grid points has $r_\perp = \sqrt{\Delta x^2 + \Delta z^2}$ (minimum-image
periodic separations) and is averaged into the column of `R_EDGES` that contains it; the first column, of width $dx$
from 0, holds only $r_\perp = 0$, the pairs on the same column of cells.

**Why the grid's $\xi$ is not isotropic.** The toy field is isotropic, yet its expectation on the grid (a) is not a
function of $\lvert\mathbf r\rvert$ alone.

(i) The grid holds the modes $\mathbf k = 2\pi\mathbf n/L$ of the cube $\mathcal K$, $\lvert k_i\rvert \le \pi/dx$, so

$$
\xi_{\rm grid}(\mathbf r) = \frac{1}{V}\sum_{\mathbf k \in \mathcal K,\ \mathbf k \neq 0} P(k)\,e^{i\mathbf k\cdot\mathbf r}.
$$

(ii) Poisson summation, $V^{-1}\sum_{\mathbf k \in 2\pi\mathbb Z^3/L} f(\mathbf k)\,e^{i\mathbf k\cdot\mathbf r} =
\sum_{\mathbf n \in \mathbb Z^3}\tilde f(\mathbf r + L\mathbf n)$ with $\tilde f$ the inverse Fourier transform of $f$,
applied to $f = P\,\mathbb 1_{\mathcal K}$, turns the discrete sum into a sum over periodic images:

$$
\xi_{\rm grid}(\mathbf r) = \sum_{\mathbf n \in \mathbb Z^3}(\xi * w)(\mathbf r + L\mathbf n) - \bar\xi_{\rm box},\qquad
w(\mathbf x) = \prod_i \frac{\sin(\pi x_i/dx)}{\pi x_i},
$$

with $w$ the transform of the cube $\mathcal K$, an anisotropic kernel of width $\sim dx$, and
$\bar\xi_{\rm box} = P(0)/V$, the removed $\mathbf k = 0$ mode (the mean of the image sum over the box). For a red
spectrum each of the two terms diverges on its own; their difference, the sum without $\mathbf k = 0$, is finite.

(iii) **Effect 1, the images.** The nearest image of $\mathbf r = r\hat{\mathbf s}$ lies at
$d_1(\hat{\mathbf s}) = \min_{\mathbf n \neq 0}\lvert r\hat{\mathbf s} + L\mathbf n\rvert$.

- Along an axis, $\hat{\mathbf s} = \hat{\mathbf y}$: $d_1 = L - r$, which at $r = L/2$ is as close as $\mathbf r$ itself.
- Along a body diagonal, $\hat{\mathbf s} = (1, 1, 1)/\sqrt3$, at $r = L/2$: $r\hat{\mathbf s} = (L/2\sqrt3)(1, 1, 1)
  \simeq 0.289L\,(1, 1, 1)$, and the nearest image, $\mathbf n = (-1, 0, 0)$, lies at
  $L\sqrt{(1 - 0.289)^2 + 2\,(0.289)^2} = L\sqrt{0.506 + 0.167} \simeq 0.82L$ (printed above).

So the image term, $\xi(d_1)$, depends on the direction.

(iv) **Effect 2, the cube kernel.** The modes stop at $\pi/dx$ along an axis but at $\sqrt3\,\pi/dx$ along a body
diagonal, so $w$ smooths $\xi$ differently along the axes and along the diagonals, on scales of a few $dx$.

(v) **The wedges.** A wedge averages over its directions. The $\mu \approx 1$ cone around the $y$ axis holds only
directions near an axis, whose images are the closest ($d_1 \simeq L - r$); the other wedges mix the directions near the
$x$ and $z$ axes with many diagonal ones, whose images lie further away. So the wedge near $\mu = 1$ departs most from
the others, and most near $L/2$.

**Wedges.** The direction of a separation (or of a wavevector) is measured by

$$
\mu = \frac{r_\parallel}{\lvert\mathbf r\rvert} = \cos\theta,
$$

the cosine of its angle $\theta$ to the line of sight: $\mu = 1$ along the line of sight, $\mu = 0$ across it.
$\mathrm{Ly}\alpha$ 3D analyses show the anisotropy in wedges of $\mu$.
The expectation of a wedge $w$ (an interval of $\mu$ of width $\Delta\mu$), step by step:

1. In the continuum, the correlation depends on the separation and its direction, $\xi(r, \mu)$, and the wedge average
   is $\xi_w(r) = \Delta\mu^{-1}\int_{\mu \in w}\xi(r, \mu)\,d\mu$. The toy field is isotropic, $\xi(r, \mu) = \xi(r)$,
   so every wedge would be the same curve.
2. On the grid, a wedge is the average over the separations of the grid with $\lvert\mathbf r\rvert$ in a bin $a$ of
   `R_EDGES` and $\mu = \lvert r_y\rvert/\lvert\mathbf r\rvert$ in the wedge $w$ (the same `MU_WEDGES` as the rays of the
   maps), and its expectation averages $\xi_{\rm grid}$ over the same separations:
   $\bar\xi_{a w} = N_{aw}^{-1}\sum_{\mathbf r \in a,\,\mu \in w}\xi_{\rm grid}(\mathbf r)$.
3. $\xi_{\rm grid}$ is not isotropic (the explanation above), so the dashed lines differ between wedges: the wedge near
   $\mu = 1$ departs most, near $L/2$.

In [ ]:
from skewerjax.utils.plotting import plot_xi_mu_wedges
fig, axs, chi2_rmu = plot_xi_mu_wedges(res, r_edges=R_EDGES, r_bao=boxes.r_bao, quantity="r2xi", y_scale="symlog")
plt.show()
print("chi2/dof per wedge (diagonal):", np.round(chi2_rmu, 2))

## Part B — Skewers

### B1 · Skewers through each box

**Layout.** In every box, skewers run along $y$ through the centres of the cell columns (the nodes $x = i_x\,dx$,
$z = i_z\,dx$ of `grf_cells`), from $y = 0$ to $y = L$, so their first and last points are the same place of the periodic
box. `SkewerBundle` gives $\tau$ of $\mathrm{Ly}\alpha$ and
Si III 1206, periodic over one box (`velocity_period=True`) and both on the $\mathrm{Ly}\alpha$ velocity units reference frame axis
(`line_frame="reference"`).

**The set-up, `SkewerRun`** (next cell, which prints what follows from it), with $p$ = `points_per_cell` and
$N_{\rm pix}$ = `n_pix`:

| Argument | What it controls | Result or cost |
|---|---|---|
| `stride` $= (s_x, s_z)$ | the cell columns that become skewers: every $s_x$-th in $x$ and every $s_z$-th in $z$, from column 0, exactly through the cell centres $x = i\,s_x\,dx$, $z = j\,s_z\,dx$; $\lceil N/s_x\rceil \lceil N/s_z\rceil$ skewers per box, $s\,dx$ apart | the result: the number of skewers (the noise of every statistic) and the transverse sampling of Part C (Nyquist frequency $\pi/(s\,dx)$); the cost is linear in the number of skewers |
| `points_per_cell` | $p$, the points of the path per cell length along $y$, $N p + 1$ per skewer: every cell is cut into pieces of width $W = dx/p$ | the result until it converges ($\tau$ sums the pieces at their centroids, accurate while $W \ll b$: the resolution check below); the cost of $\tau$ is linear in $p$ |
| `n_pix` | $N_{\rm pix}$ velocity pixels over one period, of width $dv = v_{\rm per}/N_{\rm pix}$: the velocity resolution | the result: the $k$ range of P1D, up to $\pi/dv$, and the line-of-sight resolution of Part C; the cost of $\tau$ is linear in $N_{\rm pix}$ |
| `batch` | the skewers per call of `cell_pieces` and `compute_optical_depth` | the memory only, never the result |

Here $v_{\rm per}$ is the velocity span of one box along a skewer, and $u = v_{\rm per}/L$ converts comoving distances to
velocities (Part C).

**Transmission and overflux.** $T = e^{-\tau}$ is the **transmission**; the flux is the observed $F = F_0\,T$, the
continuum $F_0$ times the transmission. The overflux follows continuum-fitting convention:

$$
\delta = \frac{F}{F_0\,\bar T} - 1 = \frac{T}{\bar T} - 1,\qquad \bar T = \langle T\rangle,
$$

the mean over a set of pixels chosen by `run_skewers(..., mean=)`. Each line $j$ has $\delta_j = T_j/\bar T_j - 1$, and the
total, $T = \prod_j T_j$, has $\bar T = \langle\prod_j T_j\rangle \neq \prod_j \bar T_j$.

- **The normalisation modes.** $\langle\delta\rangle = 0$ holds over the set of pixels of the mean:

  | `mean` | $\bar T$ over | removes | use it for |
  |---|---|---|---|
  | `"redshift"` (default) | the pixels at the same redshift, of every skewer of every box: $\bar T(z)$ | the mean-flux evolution along the skewers, a pattern shared by all of them | the observational convention, the mean flux as a function of $z$ |
  | `"global"` | every pixel of every skewer of every box: one number | only the overall mean; a drift of $\bar T$ along $v$ stays in $\delta$ | a path without evolution |
  | `"box"` | every pixel of every skewer of one box | the box-to-box scatter of the mean ($\mathbf k = 0$ of each box) | normalising each realisation by itself |
  | `"skewer"` | the pixels of one skewer | the $k_\parallel = 0$ mode of every skewer | mimicking a fitted continuum amplitude |

  All skewers share one grid of pixel scale factors here, so $\bar T(z)$ is the mean at each pixel over the ensemble.
- **The drift along $v$.** The skewers are in the exact frame: $z$ sits at $y = 0$ and $a(y)$ decreases along the box,
  by $\Delta\ln a = v_{\rm per}/c$, while the box is wrapped as periodic. A uniform gas then has
  $\tau \propto a^{-2}/(aH)$ per pixel, varying across the box, so $\bar T(v)$ drifts and
  jumps back at the wrap. `"redshift"` removes that drift from $\delta$; with `"global"` it stays as a pattern shared by
  every skewer, which shows as horizontal bands in the $(r_\perp, r_\parallel)$ map (C1).
- **The density conversion `A_EFF`.** `SkewerRun(a_eff=)` replaces $a_p^{-2}$ of every piece by one $a_{\rm eff}^{-2}$
  (e.g. $1/(1+z)$, a fixed comoving-to-physical conversion) and touches nothing else. The line centres keep
  $c\ln(a_0/a(y))$, so a column still spreads over a velocity width $\propto aH\,\ell$: the Jacobian $1/(aH)$ remains,
  about $-1.8$ % across the box ($aH$ grows with $z$).
- **The continuum.** Here $F_0$ is known (in effect $F/F_0 = T$), so $\delta = T/\bar T - 1$ exactly. In data, $F_0$
  must be estimated for each quasar. A per-quasar change in the continuum's amplitude or slope is indistinguishable from
  a large-scale fluctuation of the transmission along that sightline, so even a correct continuum model absorbs those
  modes of $\delta$; analyses correct for this with a projection. It does not arise here, because $F_0$ is known.

In [ ]:
from skewerjax.simulations import SkewerRun, run_skewers
SETUP = SkewerRun(stride=(2, 2), points_per_cell=32, n_pix=128, a_eff=(1 + boxes.z), detuning="linear")    # a_eff = None: a_p per piece (for computing physical number densities); a_eff = 1/(1+z): fixed comoving-to-physical conversion
der = SETUP.derived(boxes)
n_x, n_z = der["n_x"], der["n_z"]
cols = boxes.columns(SETUP.stride)
print(f"skewers: stride {tuple(SETUP.stride)} -> {n_x} x {n_z} = {der['per_box']} per box, {der['per_box'] * idx.size} in all "
      f"({idx.size} boxes)")
print(f"transverse spacing of the skewers: stride x dx = {SETUP.stride[0]} x {boxes.dx:.4g} = {der['d_perp_cMpc'][0]:.4g} cMpc in x")
print(f"points per skewer: {der['points_per_skewer']} ({SETUP.points_per_cell} per cell); pieces of W = {boxes.dx / SETUP.points_per_cell:.4f} cMpc "
      f"= {der['piece_kms']:.3f} km/s")
print(f"pixels: {SETUP.n_pix} over v_per = {der['period_kms']:.0f} km/s: dv = {der['dv_kms']:.2f} km/s, "
      f"P1D up to pi / dv = {np.pi / der['dv_kms']:.4f} s/km")
print(f"u = v_per / L = {der['kms_per_cMpc']:.2f} km/s per cMpc (aH = {der['aH_start']:.2f} at y = 0, {der['aH_end']:.2f} at y = L; "
      f"z = {boxes.z:g} at y = 0); density conversion: " + ("a(y)^-2 per piece" if SETUP.a_eff is None else f"a_eff = {SETUP.a_eff:.4f}"))
lat = SETUP.limits(boxes)                                             # the lattice of Part C: exactly the skewers' samples
print(f"lattice of Part C, across: spacing s dx = {lat['d_perp'][0]:.4g} (x), {lat['d_perp'][1]:.4g} (z) cMpc; k_x, k_z multiples of "
      f"2 pi / L = {lat['k_fund']:.4f} up to pi / (s dx) = {lat['k_nyq_perp'][0]:.4f} (x), {lat['k_nyq_perp'][1]:.4f} (z) cMpc^-1")
print(f"lattice of Part C, along the line of sight: spacing dv = v_per / N_pix = {lat['dv_kms']:.2f} km/s; k_par multiples of "
      f"2 pi / v_per = {lat['k_par_min_skm']:.3e} up to pi / dv = {lat['k_par_max_skm']:.3e} s/km; r_par from dv to v_per / 2 = "
      f"{lat['period_kms'] / 2:.0f} km/s")

**The bins of Part C** (next cell), passed to `run_skewers` and to the figures of Part C. The statistics use exactly
the lattice the skewers produce, with no regridding or interpolation before the FFT: $s\,dx$ across, $dv$ along the line
of sight (printed above), converted to comoving units with the one factor $u$ for the isotropic quantities (Part C).

- $r_\perp$, `RPERP_EDGES_C`: width the skewer spacing $s\,dx$, from 0 to $L/2$; the first bin holds the pairs on the same
  skewer.
- $r_\parallel$: the lattice's own separations, one row per pixel separation, $r_\parallel = j\,dv$ from 0 to
  $v_{\rm per}/2$, each row centred on its separation (no array: they follow from $N_{\rm pix}$).
- $\lvert\mathbf r\rvert$, `R_EDGES_C`, also the $\lvert\mathbf r\rvert$ bins of the wedges: width the coarser lattice
  spacing in cMpc, $\max(s\,dx, dv/u)$, from 0 to $L/2$.
- $\lvert\mathbf k\rvert$, `K_EDGES_C`: the binning of `K_EDGES` up to the smaller of the two Nyquist frequencies in
  $\mathrm{cMpc}^{-1}$, $\pi/(s\,dx)$ and $\pi u/dv$, so that every bin holds all directions.

In [ ]:
# The bins of the 3D statistics of Part C in comoving units [cMpc], [cMpc^-1] (the line of sight converted with u)
u = lat["kms_per_cMpc"]                                                           # km/s per cMpc: v = u x, k_v = k_x / u
d_min, d_max, dv_c = lat["d_perp"].min(), lat["d_perp"].max(), lat["dv_kms"] / u   # the skewer spacing s dx, a pixel dv / u [cMpc]
RPERP_EDGES_C = np.arange(0, L / 2 + d_min, d_min)                                # r_perp: width s dx to L / 2 (first bin: the same skewer)
R_EDGES_C = np.arange(0, L / 2 + max(d_max, dv_c), max(d_max, dv_c))              # |r| (and the wedges): the coarser spacing, to L / 2
K_EDGES_C = boxes.k_edges(k_max=min(np.pi / d_max, np.pi / dv_c))                 # |k|: as K_EDGES, to the smaller Nyquist frequency
EDGES_C = dict(k_edges=K_EDGES_C, r_edges=R_EDGES_C, rperp_edges=RPERP_EDGES_C)
print(f"Part C bins: r_perp {RPERP_EDGES_C.size - 1} of width {d_min:.4g} cMpc up to {RPERP_EDGES_C[-1]:.1f} cMpc; r_par one row per "
      f"pixel separation, {SETUP.n_pix // 2 + 1} rows from 0 to v_per / 2; |r| {R_EDGES_C.size - 1} of width {max(d_max, dv_c):.4g} cMpc "
      f"up to {R_EDGES_C[-1]:.1f} cMpc; |k| {K_EDGES_C.size - 1} up to {K_EDGES_C[-1]:.4f} cMpc^-1")
print(f"full coverage of the lattice: |k| <= min(pi / (s dx), pi u / dv) = min({np.pi / d_max:.4f}, {np.pi / dv_c:.4f}) cMpc^-1, "
      f"|r| <= L / 2 = {L / 2:.1f} cMpc")

**Two passes, in memory** (`run_skewers`). First, each box is generated and $\tau$ computed along its skewers; its
$\tau$ is kept, with the densities of the cells the skewers cross. Then the mean transmission of the ensemble, one
number per line ($\bar T_j$) and for the total ($\bar T$) over every pixel of every skewer of every box,
normalises the overfluxes of every box, which are measured: the 3D statistics of Part C and the 1D statistics of Part D.
The densities, sampled at the points of the skewers, go through the same estimators. Only $\tau$, the
densities along the skewers and the statistics are held, the fields are deleted.

In [ ]:
t0 = time.perf_counter()
run = run_skewers(boxes, SETUP, edges=EDGES_C, realisations=REALISATIONS)  # every box, in memory, the bins of Part C
t_skewers = time.perf_counter() - t0
n_box, n_sk = run["flux"]["n_real"], len(cols)
dev = {d: run["time_s"][run["device"] == d].sum(1) for d in np.unique(run["device"])}
period = float(run["skewers"][0]["period"])
lines = LineBundle(list(LINE_IDS), ref_line_id=LINE_IDS[0])
b_T0 = {lid: np.sqrt(2 * phys.kB * boxes.T0 / m) / 1e3 for lid, m in zip(LINE_IDS, np.asarray(lines.mass_amu))}
print(f"{n_box} boxes x {n_sk} skewers ({n_x} x {n_z}) in {t_skewers:.0f} s")
print("time per box: " + ", ".join(f"{d} {t.mean():.0f} s ({t.size} boxes)" for d, t in dev.items())
      + f"; per box: fields {run['time_s'][:, 0].mean():.1f} s, pieces {run['time_s'][:, 1].mean():.1f} s, "
      f"tau {run['time_s'][:, 2].mean():.0f} s, statistics {run['time_s'][:, 3].mean():.1f} s")
print(f"axis: period {period:.0f} km/s, {SETUP.n_pix} pixels of dv = {period / SETUP.n_pix:.2f} km/s; a cell spans "
      f"{period / boxes.n_cells:.1f} km/s ({SETUP.n_pix / boxes.n_cells:g} pixels), a piece {period / boxes.n_cells / SETUP.points_per_cell:.2f} "
      f"km/s; b at T0: " + ", ".join(f"{lid} {b:.1f} km/s" for lid, b in b_T0.items()))
res_check = run["resolution"]
print(f"resolution check: max W / b = {np.round(res_check[..., 0].max(axis=(0, 1)), 3)}, column fraction above b/4 at most "
      f"{np.round(res_check[..., 1].max(axis=(0, 1)), 6)} (Lya, Si III); warnings: {sum(len(w) for w in run['warnings'].values())}")
T_v = run["mean_transmission_per_pixel"]                                 # T_bar(v) at each pixel over the ensemble (3, n_pix)
drift = T_v / T_v.mean(axis=1, keepdims=True) - 1                        # the drift along v of each line and the total
print(f"mean transmission (mean={run['mean']!r}): over every pixel Lya {T_v[0].mean():.4f}, Si III {T_v[1].mean():.4f}, total "
      f"{T_v[2].mean():.4f} (scatter of the box means {run['mean_F'][:, 0].std():.1e}); pixels on one redshift grid, a from "
      f"{run['scale_factor'][0]:.5f} to {run['scale_factor'][-1]:.5f}")
print("drift of T_bar(v) along v, T_bar(v) / <T_bar> - 1 (diagnostic): " + ", ".join(
      f"{n} {d[:5].mean():+.4f} -> {d[-5:].mean():+.4f} (rms {d.std():.1e})" for n, d in zip(("Lya", "Si III", "total"), drift))
      + f"; Delta ln a = v_per / c = {period / phys.c_kms:.4f}")
print(f"normalisation: max |<delta>| over the sets of mean={run['mean']!r} = {run['mean_delta_level'].max():.1e} (Lya, Si III, "
      f"total); C = T_bar / (T_bar_1 T_bar_2) - 1 = {run['C']:.4e} (mean), |C - <delta_1 delta_2>| = {run['C_error']:.1e}; "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {run['decomposition_error']:.1e}")

### B2 · One skewer, from the gas to the flux contrast

**Where the skewers run.** The H I density of realisation 0 in the plane of A2 (the first cell layer along the third
axis, $i_z = 0$), a plane that contains the line of sight and the first row of skewers: the skewers of that row are the
thin black lines, with small arrows along their direction ($+y$), $s_x\,dx$ apart (printed in B1, marked at the top).
The thick line is the skewer of the next figure.

In [ ]:
from skewerjax.utils.plotting import plot_skewer_slice, plot_toy_skewer
sk = run["skewers"][0]
on_z0 = np.flatnonzero(sk["columns"][:, 1] == 0)
j = int(on_z0[np.argmin(np.abs(sk["columns"][on_z0, 0] - boxes.n_cells // 2))])   # the column nearest to x = L/2 with i_z = 0
ix, iz = sk["columns"][j]
fig, ax, sl_info = plot_skewer_slice(fields0, L_BOX, sk["columns"], z_index=0, highlight=(ix, iz))
plt.show()
print(f"{sl_info['n_row']} skewers in the layer i_z = 0, {sl_info['spacing_cMpc']:.4g} cMpc apart; the skewer of the next figure: "
      f"x = {ix * boxes.dx:.1f} cMpc")

**Along the skewer.** Rows (a) and (b) are along the comoving position $y$ (the $N$ cell values, as steps); rows (c) and
(d) along the velocity of the $\mathrm{Ly}\alpha$ axis, on which Si III is evaluated at $v + \Delta v$ and wrapped into
the period. The overdensities (b) of H I and Si III are identical by construction (one gas, with one temperature);
their absorption is not: Si III is narrower ($b \propto m^{-1/2}$) and displaced by the separation $\Delta v$ of the two
lines, wrapped into the period (printed below). Row (c) is the transmission $T = e^{-\tau}$ of each line and the total
$T = \prod_j T_j$; row (d) their overfluxes, $\delta = T/\bar T - 1$, each against the mean transmission of the
ensemble at its redshift (B1), so the mean of (d) along this one skewer need not vanish. The very thin grey vertical lines are the grids:
in (a) and (b) the points of the path ($p$ per cell), in (c) and (d) the pixel centres, where $\tau$ is evaluated. When
the lines of a panel would be too dense to read, they are drawn only in a zoomed inset, whose range is marked on the
panel.

In [ ]:
n_col = {sp: fields0["n"][sp][ix, :, iz] for sp in ("HI", "SiIII")}
fig, axs, grid_info = plot_toy_skewer(L_BOX, n_col, sk["v"], sk["tau"][:, j], mean_transmission=run["mean_transmission"], points_per_cell=SETUP.points_per_cell)
add_panel_letters(axs)
plt.show()
dv_12 = float(lines.delta_v[1])
dv_fold = abs((dv_12 + period / 2) % period - period / 2)                    # the separation folded into half the period
print(f"Si III - Lya separation Delta v = {dv_12:.0f} km/s, wrapped into the period of {period:.0f} km/s: "
      f"{(dv_12 + period / 2) % period - period / 2:.0f} km/s; b(Lya) / b(Si III) = {b_T0[LINE_IDS[0]] / b_T0[LINE_IDS[1]]:.2f}")

## Part C — 3D statistics of the overfluxes

**Two kinds of axis.** Along the line of sight the coordinate is a **velocity**, $v$ [km/s]; across it, the coordinates
are comoving **distances**, $x_\perp$ [cMpc]. One approximate factor relates them, the mean Hubble rate along a skewer
($dv/dy = aH$ changes along it with $a(y)$):

$$
\Delta v \simeq u\,\Delta x,\qquad u = \frac{1}{L}\int_0^L aH\,dy = \frac{v_{\rm per}}{L}.
$$

Two rules follow:

- **Maps** ($r_\parallel$ against $r_\perp$): each axis keeps its native coordinate. The bottom axis is the
  perpendicular one, in cMpc; the left axis is the line of sight, in km/s.
  For reference only, a top axis gives the perpendicular
  coordinate in km/s and a right axis the line of sight in cMpc, both through $u$.
- **Isotropic quantities that combine both axes** ($\lvert\mathbf r\rvert$, $\lvert\mathbf k\rvert$, $\mu$, $P$, etc.):
  the line of sight is converted to comoving distance, $r_\parallel = v/u$ and $k_\parallel = u\,k_v$, and they are
  reported in comoving units: $r$ in cMpc, $r^2\xi$ in $\mathrm{cMpc}^2$, $k$ in $\mathrm{cMpc}^{-1}$ (the top axes give km/s and s/km for
  reference). On the lattice a pixel $dv$ spans $dv/u = L/N_{\rm pix}$ cMpc, so the lattice is a cube of side $L$.

**What the lattice resolves.** The statistics are computed on exactly the lattice of the skewers, with no regridding.
Along the line of sight: from the pixel size $dv$ up to the span of the box, $v_{\rm per}$. Across it:
from the skewer spacing $s\,dx$ (set by `stride`) up to $L$.

**Errors.** Points with error bars are the mean over the boxes and its error, as in Part A.


### C1 · The $(r_\perp, r_\parallel)$ plane of the total overflux

$\xi(r_\perp, r_\parallel)$ of the total overflux. The map is on the lattice's own bins: one row per pixel
separation, $r_\parallel = j\,dv$ from 0 to $v_{\rm per}/2$, each centred on it, and one column per bin of
`RPERP_EDGES_C` (width $s\,dx$), up to $L/2$. Every pair of lattice points, with $r_\perp = \sqrt{\Delta x^2 + \Delta z^2}$,
is averaged into the cell that contains it.

In [ ]:
from skewerjax.utils.plotting import echo_curves, echo_scales, fold_periodic, plot_overflux_3d, plot_overflux_rpar_rperp
ax3, f3 = run["axes3d"], run["flux3d"]                                   # comoving bins [cMpc], [cMpc^-1]
lim = SETUP.limits(boxes)
NAMES = {"HI-LyA": r"Ly$\alpha$", "SiIII-1206": "Si III 1206", "total": "total"}
tot = dict(f3["total"], axes=ax3)                                        # the total overflux and its bins
fig, ax, curves = plot_overflux_rpar_rperp(tot, kms_per_cMpc=u, rperp_edges=RPERP_EDGES_C, r_bao=boxes.r_bao, delta_v=dv_fold, n_sigma=1, color_scale="symlog", quantity="r2xi")
plt.show()
r_par_max = ax3["rpar"][-1]                                              # the last row: half the period [cMpc]
print(f"map: {ax3['rpar_edges'].size - 1} rows, one per pixel separation j dv (dv = {u * np.diff(ax3['rpar_edges'])[0]:.2f} km/s), "
      f"from 0 to {u * r_par_max:.0f} km/s = v_per / 2; {ax3['rperp_edges'].size - 1} columns of r_perp of width "
      f"{np.diff(ax3['rperp_edges'])[0]:.3g} cMpc up to {ax3['rperp_edges'][-1]:.1f} cMpc; u r_BAO = {u * boxes.r_bao:.0f} km/s")
DELTA = dv_fold / u                                                      # the line separation in comoving units [cMpc]
ends = fold_periodic([boxes.r_bao - DELTA, boxes.r_bao + DELTA], L_BOX)
print(f"Delta = Delta v / u = {DELTA:.2f} cMpc ({dv_fold:.0f} km/s); the echo branches start on the line of sight at "
      f"|r_BAO - Delta| = {ends[0]:.1f} and r_BAO + Delta = {ends[1]:.1f} cMpc (folded into [0, L / 2]) and meet at "
      f"(r_perp, r_par) = ({boxes.r_bao:g}, {DELTA:.1f}) cMpc")
X, Xe = f3["total"]["xi_map"], f3["total"]["xi_map_err"]
ip = int(np.argmin(np.abs(ax3["rpar"] - DELTA)))                         # the r_par row of Delta
for q in (0, 1):                                    # r_perp columns: 0 = pairs on the same skewer, 1 = one transverse spacing
    excess = X[ip, q] - 0.5 * (X[ip - 2, q] + X[ip + 2, q])
    print(f"Si III - Lya at r_par = Delta, r_perp bin {q} ({ax3['rperp_edges'][q]:.1f}-{ax3['rperp_edges'][q + 1]:.1f} cMpc): "
          f"xi = {X[ip, q]:.2e}, excess over the rows two pixels away {excess:.2e} = {excess / Xe[ip, q]:.1f} sigma")
b_ratio = np.sqrt(f3["SiIII-1206"]["P_iso"][0] / f3["HI-LyA"]["P_iso"][0])  # b_S / b_L from the spectra at the lowest |k|
R2 = np.nan_to_num(ax3["rpar_map"]) ** 2 + np.nan_to_num(ax3["rperp_map"]) ** 2
M, Me = R2 * X, R2 * Xe                                                  # r^2 xi of the map and its error [cMpc^2]

def along(c):                                       # the mean of r^2 xi over the cells a curve crosses, and its error
    i = np.clip(np.searchsorted(ax3["rpar_edges"], np.asarray(c["r_par"]) / u) - 1, 0, M.shape[0] - 1)
    k = np.clip(np.searchsorted(ax3["rperp_edges"], np.asarray(c["r_perp"])) - 1, 0, M.shape[1] - 1)
    cells = np.unique(np.stack([i, k]), axis=1)
    return M[cells[0], cells[1]].mean(), np.sqrt(np.sum(Me[cells[0], cells[1]] ** 2)) / cells.shape[1]

for name in (r"$r_{\rm BAO}$", r"echo $+\Delta$", r"echo $-\Delta$"):
    m, e = along(curves[name])
    print(f"  mean r^2 xi along the {name.replace('$', '').replace(chr(92) + 'rm ', '')} curve: {m:+.3f} +- {e:.3f} cMpc^2")
print(f"b_S / b_L = sqrt(P_SiIII / P_Lya) at the lowest |k| = {b_ratio:.3f}: the expected amplitude of the echoes relative to "
      f"the ring")
shared = run["mean_delta"][-1]                                           # <delta_tot> at each pixel, over every skewer of every box
s_glob = drift[-1] / (1 + drift[-1].mean())                              # the pattern a global mean would keep: T_bar(v) / <T_bar> - 1
print(f"pattern shared by every skewer in delta_tot (mean={run['mean']!r}, a diagnostic): rms {shared.std():.1e} "
      f"(zero by construction with 'redshift'); with mean='global' it would be the drift of T_bar(v), rms {s_glob.std():.1e}, "
      f"adding about {s_glob.var():.1e} to xi at every r_perp (the k_perp = 0 modes)")

**The echoes of the BAO ring.** On large scales the overflux of each line follows the gas, $\delta_j \simeq b_j\delta_g$,
with a bias $b_j$. Si III is evaluated on the $\mathrm{Ly}\alpha$ axis at $v + \Delta v$, so it samples the gas a
distance $\Delta = \Delta v/u$ further along the line of sight:

$$
\delta_{\rm tot}(\mathbf x) \simeq b_L\,\delta_g(\mathbf x) + b_S\,\delta_g(\mathbf x + \Delta\hat{\mathbf y}).
$$

Its correlation, with $\xi_g$ that of the gas:

$$
\xi_{\rm tot}(r_\perp, r_\parallel) \simeq \big(b_L^2 + b_S^2\big)\,\xi_g\Big(\sqrt{r_\perp^2 + r_\parallel^2}\Big)
+ b_Lb_S\Big[\xi_g\Big(\sqrt{r_\perp^2 + (r_\parallel - \Delta)^2}\Big) + \xi_g\Big(\sqrt{r_\perp^2 + (r_\parallel + \Delta)^2}\Big)\Big].
$$

The cross terms copy the gas correlation shifted by $\pm\Delta$ along the line of sight: its peak at $r = 0$ becomes the
spot at $(0, \Delta)$ (the diamond), and its BAO ring two rings centred on $(0, \pm\Delta)$. After folding
$r_\parallel \to \lvert r_\parallel\rvert$, both lie on the BAO ring re-centred on the Si III spot:

$$
r_\perp^2 + (r_\parallel - \Delta)^2 = r_{\rm BAO}^2,\qquad
r_\parallel^{\pm}(r_\perp) = \Big\lvert\sqrt{r_{\rm BAO}^2 - r_\perp^2} \pm \Delta\Big\rvert,\quad r_\perp \le r_{\rm BAO}.
$$

- **Ends.** On the line of sight, $r_\perp = 0$, the branches start at $r_\parallel = r_{\rm BAO} \pm \Delta$ (the $-$
  branch at $\lvert r_{\rm BAO} - \Delta\rvert$); they meet at $(r_\perp, r_\parallel) = (r_{\rm BAO}, \Delta)$.
- **Periodicity.** Line-of-sight separations are periodic with period $v_{\rm per}$ ($L$ in cMpc), so every curve, the
  ring included, is folded with $y \to \min\big(y \bmod v_{\rm per},\ v_{\rm per} - (y \bmod v_{\rm per})\big)$: a branch
  that passes beyond $v_{\rm per}/2$ re-enters closer to 0. Nothing is drawn beyond $r_\perp = L/2$.
- **Amplitude.** The ring carries $(b_L^2 + b_S^2)\,\xi_g$, each echo $b_Lb_S\,\xi_g$: relative to the ring, the echoes
  are of order $b_S/b_L$ (for $b_S \ll b_L$), with $b_S/b_L \simeq \sqrt{P_{\rm SiIII}/P_{{\rm Ly}\alpha}}$ at the
  lowest $\lvert\mathbf k\rvert$ (printed).

### C2 · Power spectrum and correlation function of the overfluxes

- **$\lvert\mathbf k\rvert \le k_{\rm N} = \min\bigl(\pi N_{\rm pix}/L,\ \pi/(s_x\,dx),\ \pi/(s_z\,dx)\bigr)$**, in the
  bins `K_EDGES_C` (B1), the smallest Nyquist frequency of the lattice ($\pi N_{\rm pix}/L = u\,\pi/dv$ along the line of sight). A
  $\lvert\mathbf k\rvert$ bin averages a shell of modes, and only inside the sphere of radius $k_{\rm N}$ does every
  shell hold modes in every direction. Beyond it, the coarser axes have no modes left, so the shell keeps only the
  directions near the finer axis, and $P(\lvert\mathbf k\rvert)$ would turn into the power along that one direction.
- **$\lvert\mathbf r\rvert \le L/2$**, in the bins `R_EDGES_C` (B1). The lattice is periodic, so is its correlation function,
  $\xi(\mathbf r + L\hat{\mathbf e}_i) = \xi(\mathbf r)$: a separation beyond half the box along an axis is the same as
  its image inside, and adds nothing new. Below the transverse spacing the $\lvert\mathbf r\rvert$ bins hold separations
  along the line of sight only.

In [ ]:
from skewerjax.simulations import bao_height
fig, axs = plot_overflux_3d(f3, ax3, kms_per_cMpc=u, r_bao=boxes.r_bao, r_marks={r"$\Delta$": float(fold_periodic(DELTA, L_BOX))},
                            names=NAMES, k_edges=K_EDGES_C, r_edges=R_EDGES_C, limits=lim, quantity="r2xi", y_scale="symlog", k_scale="linear")
add_panel_letters([axs["P"], axs["xi"]])
plt.show()
PT = f3["total"]["P_iso"]
print(f"lattice {n_x} x {SETUP.n_pix} x {n_z} per box, a cube of side L = {L_BOX:g} cMpc (v_per = {period:.0f} km/s along the line of sight)")
print(f"|k| from {ax3['k_edges'][0]:.3f} to k_N = {ax3['k_edges'][-1]:.3f} cMpc^-1 in {ax3['k_edges'].size - 1} bins: Nyquist "
      f"pi N_pix / L = {lim['k_nyq_par']:.3f} cMpc^-1 along the line of sight (pi / dv = {lim['k_nyq_par'] / u:.2e} s/km), "
      f"pi / (s dx) = {lim['k_nyq_perp'][0]:.3f} (x), {lim['k_nyq_perp'][1]:.3f} (z) cMpc^-1 across it")
print(f"|r| up to L / 2 = {lim['r_max']:.0f} cMpc in {ax3['r_edges'].size - 1} bins of {np.diff(ax3['r_edges'])[0]:.2f} cMpc; "
      f"line-of-sight separations only below the transverse spacing {lim['r_perp_min']:.2f} cMpc")
print("P_line / P_tot at the lowest |k|: " + ", ".join(f"{f} {f3[f]['P_iso'][0] / PT[0]:.2f}" for f in LINE_IDS))
r3 = ax3["r_iso"]
d3, d3_th = run["density3d"]["total"], run["density_theory"]          # the density on the same skewers (Part E)
bao_rho = bao_height(r3, r3**2 * d3_th["xi_iso"], boxes.r_bao, boxes.sigma_bao)["height"]
print(f"BAO peak of r^2 xi above its sidebands at r_BAO -+ 4 sigma_BAO [cMpc^2], against (P_X / P_density at the lowest |k|) "
      f"x the density's expected {bao_rho:.3f}:")
for f in (*LINE_IDS, "total"):
    bf = bao_height(r3, r3**2 * f3[f]["xi_iso"], boxes.r_bao, boxes.sigma_bao, r3**2 * f3[f]["xi_iso_err"])
    print(f"  {NAMES[f]:>12}: r^2 xi(r_BAO) = {bf['value']:.3f}, height {bf['height']:.3f} +- {bf['height_err']:.3f}; "
          f"scaled density {f3[f]['P_iso'][0] / d3['P_iso'][0] * bao_rho:.3f}")

### C3 · Wedges of the total overflux

In [ ]:
ECHO_SCALES = echo_scales(boxes.r_bao, DELTA, L_BOX)                     # the scales of C1 [cMpc], folded into [0, L / 2]
fig, axs, _ = plot_xi_mu_wedges(tot, r_edges=R_EDGES_C, r_marks=ECHO_SCALES, kms_per_cMpc=u, quantity="r2xi", y_scale = "linear",)
plt.show()
mu_e, r_mu = ax3["mu_edges"], ax3["r_mu"]
jb = np.nanargmin(np.abs(r_mu - boxes.r_bao), axis=1)
print("characteristic scales [cMpc]: " + ", ".join(f"{k.replace('$', '')} {v:.1f}" for k, v in ECHO_SCALES.items()))
print("r^2 xi of the total at r_BAO [cMpc^2]: " + "; ".join(
      f"{mu_e[w]:.2f} < mu < {mu_e[w + 1]:.2f}: {r_mu[w, j]**2 * f3['total']['xi_mu'][w, j]:+.2f} +- {r_mu[w, j]**2 * f3['total']['xi_mu_err'][w, j]:.2f}"
      for w, j in enumerate(jb)))

## Part D — 1D statistics

### D1 · P1D next to a DESI measurement

In [ ]:
from skewerjax.utils.plotting import plot_p1d_reference
from skewerjax.utils.reference_data import P1D_DESI_DR1_Z22 as P1D_REF
fig, ax = plot_p1d_reference(run["p1d"], P1D_REF, names=NAMES, k_max=P1D_REF["k_max_figure"], kms_per_cMpc=u)
plt.show()
kc = run["p1d"]["k"]
toy = np.interp(P1D_REF["k"], kc, kc * run["p1d"]["p1d"][0] / np.pi)
print(f"toy z = {boxes.z:g}, reference z = {P1D_REF['z']:g}; reference / toy (Lya) over the reference's k range: "
      f"median {np.median(P1D_REF['kP_pi'] / toy):.0f}")
print(f"Si III - Lya separation Delta v = {abs(dv_12):.0f} km/s: period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.4f} s/km; "
      f"Nyquist frequency of the cells along the skewers pi / (u dx) = {np.pi / (u * boxes.dx):.4f} s/km")

### D2 · The decomposition by order

**The terms.** `DeltaFieldBuilder` also forms, per skewer, the products $\delta_A = \prod_{j \in A}\delta_j$ of every
subset $A$ of lines. With $\mathcal C$, the means of the products over the set of pixels of the mean transmission, $\mathcal C = \langle\delta_1\delta_2\rangle$
for two lines (plus the higher orders for more), the total overflux decomposes **exactly** into them (review notes, Sec.
Overflux, with the mean of the ensemble):

$$
\delta_{\rm tot} = \frac{\sum_j\delta_j + \sum_{j_1 < j_2}\delta_{j_1}\delta_{j_2} + \cdots - \mathcal C}{1 + \mathcal C},\qquad
\mathcal C = \sum_{\lvert A\rvert \ge 2}\langle\delta_A\rangle = \frac{\bar T}{\prod_j \bar T_j} - 1.
$$

It follows from $T = \prod_j \bar T_j(1 + \delta_j)$ and its mean over the same set,
$\bar T = \prod_j \bar T_j\,\bigl(1 + \sum_j\langle\delta_j\rangle + \mathcal C\bigr)$. The order-1 term of the
denominator, $\sum_j\langle\delta_j\rangle$, vanishes because every $\bar T_j$ and $\bar T$ are
means over the same set of pixels (every mode of `mean`). With the default `"redshift"`, $\mathcal C(v)$ is
$\langle\delta_1\delta_2\rangle$ at each pixel over the ensemble (one number with `"global"`, one per box with `"box"`,
one per skewer with `"skewer"`; B1 prints the largest difference, at machine precision), and the sum of the terms
reproduces $\delta_{\rm tot}$ at every pixel of every skewer to machine precision (printed).
`OverfluxStatistics1D` gives P1D and $\xi_{\rm 1D}$ of the total, of every pair of terms $(A, B)$ and of every order
$\lvert A\rvert + \lvert B\rvert$: order $n$ collects the terms with $\lvert A\rvert + \lvert B\rvert = n$, order 0 is the
normalisation term $\mathcal C$, and the orders add up to the total to machine precision (printed).

**The terms of two lines.** With two lines, at each pixel,

$$
T = \bar T_1 \bar T_2\, (1 + \delta_1)(1 + \delta_2),\qquad
\delta = \frac{\delta_1 + \delta_2 + \delta_1\delta_2 - \mathcal C}{1 + \mathcal C},\qquad \mathcal C = \langle \delta_1 \delta_2 \rangle.
$$

Every term of $\xi_{\rm 1D}$ (real space) or P1D (Fourier space) is a pair of subsets $A, B$ of $\{1, 2\}$ (or the
normalisation term), of order $\lvert A\rvert + \lvert B\rvert$:

| $A \backslash B$ | $\mathcal{C}$ (order 0) | 1 (order 1) | 2 (order 1) | 12 (order 2) |
|---|---|---|---|---|
| $\mathcal{C}$ | $+\mathcal{C}\mathcal{C}$ | 0 | 0 | $-\mathcal{C}\mathcal{C}$ |
| 1 | 0 | $\xi_{11}$, $P_{11}$ | $\xi_{21}$, $P_{21}$ | $\xi_{(12)1}$, $P_{(12)1}$ |
| 2 | 0 | $\xi_{12}$, $P_{12}$ | $\xi_{22}$, $P_{22}$ | $\xi_{(12)2}$, $P_{(12)2}$ |
| 12 | $-\mathcal{C}\mathcal{C}$ | $\xi_{1(12)}$, $P_{1(12)}$ | $\xi_{2(12)}$, $P_{2(12)}$ | $\xi_{(12)(12)}$, $P_{(12)(12)}$ |

The zeros are exact after the ensemble average, because $\langle\delta_j\rangle = 0$ over the set of the mean
(here at each pixel): a pair $(\mathcal C, j)$ is $-\mathcal C$ times the mean of $\delta_j$ along the skewer in
$\xi_{\rm 1D}$, and lives at $k = 0$ alone in P1D. The ensemble average gives

$$
\xi = \frac{-\mathcal{C}^2 + \xi_{11} + \xi_{22} + \xi_{12,{\rm sym}} + \xi_{1(12),{\rm sym}} + \xi_{2(12),{\rm sym}} + \xi_{(12)(12)}}{(1 + \mathcal C)^2},
$$

$$
P(k) = \frac{P_{11} + P_{22} + P_{12,{\rm sym}} + P_{1(12),{\rm sym}} + P_{2(12),{\rm sym}} + P_{(12)(12)} - 2\pi\,\mathcal C^2\, \delta_D(k)}{(1 + \mathcal C)^2},
$$

where "sym" adds the pair and its transpose: 1/2 with 2/1, 1/12 with 12/1, and 2/12 with 12/2 (written for a constant
$\mathcal C$). With the default `"redshift"`, $\mathcal C(v)$ varies slowly along the skewer, with the mean
transmission, and the code weights every term at its own pixel by $1/(1 + \mathcal C(v))$, which keeps the
decomposition exact.

For a second line that is a shifted copy of the first, $\delta_2(v) \propto \delta_1(v - \Delta v)$, the cross term is
$P_{12,{\rm sym}} \propto 2\cos(k\Delta v)\, P_{11}$. It oscillates with period $2\pi/\Delta v$ in $k$ (D1, and
the cross terms of D3): the Fourier image of the $\xi_{\rm 1D}$ peak at $\Delta v$.

**By order.** Order 2 holds the Gaussian-like products and dominates; orders 3 and 4 are the non-linear corrections of
the product of the two fields. The line separation $\Delta v$, folded into the period (dashed; B2 prints both), is where
the cross terms put a peak in $\xi_{\rm 1D}$; in P1D they oscillate as $\cos(k\Delta v)$, with period $2\pi/\Delta v$
in $k$ (printed below in units of the mode spacing). The top axes give $k_\parallel$ and the separation in comoving units
(factor $u$).

In [ ]:
from skewerjax.utils.plotting import plot_orders_compared, plot_terms_compared
ens = {"total overflux": run["flux"]}
dv_lines = [0.0, dv_12]
fig, axs = plot_orders_compared(ens, "P", delta_v_list=dv_lines, kms_per_cMpc=u)
add_panel_letters(axs.T.ravel())
plt.show()
fig, axs = plot_orders_compared(ens, "xi", delta_v_list=dv_lines, kms_per_cMpc=u)
add_panel_letters(axs.T.ravel())
plt.show()
e = run["flux"]
print(f"max |sum of the orders - total| / max |total|: P {np.abs(e['P_order'].sum(0) - e['P_total']).max() / np.abs(e['P_total']).max():.1e}, "
      f"xi {np.abs(e['xi_order'].sum(0) - e['xi_total']).max() / np.abs(e['xi_total']).max():.1e}; at every pixel: "
      f"max |[sum_A delta_A - C] / (1 + C) - delta_tot| = {run['decomposition_error']:.1e}; order 1 / max |total|: "
      f"P {np.abs(e['P_order'][1]).max() / np.abs(e['P_total']).max():.1e}")
print(f"cos(k Delta v): period 2 pi / Delta v = {2 * np.pi / abs(dv_12):.2e} s/km = "
      f"{period / abs(dv_12):.2f} mode spacings (2 pi / period)")

### D3 · Each order, term by term

The same decomposition one level down, one row per order. Every term $(A, B)$ of an order is drawn with its transpose
$(B, A)$ as A–B (their sum, which is real), on the order's total (the thick line). Order 2 holds the products 1–1, 2–2
and 1–2 (the cross term) and the normalisation cross term $\mathcal{C}$–12; orders 3 and 4 are 1–12, 2–12 and 12–12. The
order-0 term $\mathcal{C}$–$\mathcal{C}$ is not shown: it is nearly constant in $\xi_{\rm 1D}$, and lives at the lowest
$k$ in P1D.

In [ ]:
fig, axs = plot_terms_compared(ens, "P", kms_per_cMpc=u)
add_panel_letters(axs.ravel())
plt.show()
fig, axs = plot_terms_compared(ens, "xi", kms_per_cMpc=u)
for ax in axs.ravel():
    ax.axvline(dv_fold, color=DIM, ls=":", lw=1.2)                       # the separation, folded
add_panel_letters(axs.ravel())
plt.show()

*Reading the figures.* Every term follows the density field, scaled (at linear order the
overflux is $-\bar\tau$ times the density contrast, a bias); the thermal widths of both lines (B1) are narrower than a pixel, so their
cut-off lies beyond the $k$ range. The cross terms 1–2 carry the line separation, a property of the axis, not of the
gas: a peak at $\Delta v$ in $\xi_{\rm 1D}$, an oscillation of period $2\pi/\Delta v$ in P1D. With zero velocities
(`velocity="zero"`) the flux is in real space, apart from the thermal widths; `velocity="linear"` moves the absorption by
the peculiar velocities (redshift space).

The kernels of `skewerjax.summary_stats` (`p1d_kernel`, `GridStatistics3D`, `binned_mean`) are jitted; the checks
are in `tests/summary_stats/test_statistics.py` and `tests/simulations/test_toy_boxes.py`.

## Part E — statistics of the density on the skewers

The skewer run samples the gas along every skewer, one-to-one with the overflux skewers of Part C. Part E puts the
density overdensity on those skewers through exactly the figures of Part C (the same functions, options, conventions and
transverse bins), so each figure of E can be compared with its counterpart in C: what the overflux adds or removes with
respect to the gas it traces.

- **The fields.** The overdensities of $n_{\rm HI}$, of $n_{\rm SiIII}$ and of their sum, each against its global mean
  over every point of every skewer of every box (the density has no evolution along $y$; the overflux uses $\bar T(z)$
  by default, Concepts),
  $\delta_X = n_X/\langle n_X\rangle - 1$. In this toy model $n_{\rm SiIII}$ is `siiii_to_hi` times $n_{\rm HI}$, so the
  three are identical (printed); their curves coincide.
- **The line of sight is a comoving distance**, in cMpc, sampled at the points of the skewers, spacing
  $dx/p$ ($p$ = `points_per_cell`), the duplicated periodic end point dropped, so the lattice is exactly periodic; the
  gas is uniform in each cell, so each point takes the value of its cell. Its own separations and modes are the
  line-of-sight bins; across, the bins of Part C (`RPERP_EDGES_C`, `R_EDGES_C`, `K_EDGES_C`, in cMpc). The reference axes
  give km/s through $u$, for the comparison with C.
- **The expectation.** Unlike the overflux, the density has an exact expectation on this lattice: $\xi_{\rm grid}$ at its
  separations (linear between the cells along the line of sight) and the transversely aliased spectrum
  (`lattice_density_theory`), drawn dashed.
- **What should match, and what should not.** The density carries the BAO ring and the large-scale shape of the
  overflux, scaled by $b_X^2$ (C2), but none of the features of the Si III–$\mathrm{Ly}\alpha$ cross term: no spot at
  $\Delta$ and no echoes (marked at the same places as in C), since the density of both species is the same gas at the
  same place.

In [ ]:
from skewerjax.simulations import DENSITY_FIELDS, lattice_density_check
axE, dE, thE = run["density_axes3d"], run["density3d"], run["density_theory"]   # the density at the points of the skewers
NAMES_E = {"HI": r"$n_{\rm HI}$", "SiIII": r"$n_{\rm SiIII}$", "total": r"$n_{\rm HI} + n_{\rm SiIII}$"}
n_los = SETUP.points_per_cell * boxes.n_cells                               # points along a skewer, end point dropped
LIM_E = dict(k_min=lim["k_min"], k_max=min(np.pi / lim["d_perp"].max(), np.pi * n_los / L_BOX),
             r_min=max(lim["d_perp"].max(), L_BOX / n_los), r_max=L_BOX / 2)  # the full coverage of this lattice
print(f"lattice: {n_x} x {n_los} x {n_z} points per box, spacing {L_BOX / n_los:.4f} cMpc = dx / p along the line of sight "
      f"({u * L_BOX / n_los:.2f} km/s), {lim['d_perp'][0]:.4g} cMpc across")
print("global means <n_X> / n_bar_X: " + ", ".join(f"{f} {run['density_mean'][f]:.6f}" for f in DENSITY_FIELDS)
      + f"; identical overdensities (n_SiIII = r n_HI): max |delta_HI - delta_SiIII| = {run['density_identity'][0]:.1e}, "
      f"max |delta_HI - delta_total| = {run['density_identity'][1]:.1e}")

### E1 · The $(r_\perp, r_\parallel)$ plane of the density

The counterpart of C1 for the total density, with the same options, curves and markers; the line of sight in cMpc (km/s
on the right), one row per separation of the points, and the expected contours dashed.

In [ ]:
densE = dict(dE["total"], axes=axE)
fig, ax, curves_E = plot_overflux_rpar_rperp(densE, kms_per_cMpc=u, rperp_edges=RPERP_EDGES_C, r_bao=boxes.r_bao, delta_v=dv_fold,
                                             n_sigma=1, color_scale="symlog", unit="cMpc", theory=thE)
plt.show()
XE, XEe = dE["total"]["xi_map"], dE["total"]["xi_map_err"]
ipE = int(np.argmin(np.abs(axE["rpar"] - DELTA)))                           # the r_par row of Delta
step = SETUP.points_per_cell * 2                                             # two cells away, as in C1 (two pixels)
exE = XE[ipE, 0] - 0.5 * (XE[ipE - step, 0] + XE[ipE + step, 0])
print(f"map: {axE['rpar_edges'].size - 1} rows, one per separation j dx / p from 0 to L / 2; at r_par = Delta on the same "
      f"skewer, the excess over the rows two cells away is {exE / XEe[ipE, 0]:+.1f} sigma (C1: the Si III spot)")